<div style="background: linear-gradient(135deg, #0A3D62 0%, #3C6382 100%);
            color: white; padding: 26px 30px; border-radius: 14px;
            font-family: system-ui, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;
            margin-bottom: 6px; box-shadow: 0 4px 14px rgba(10,61,98,0.25);">
  <h1 style="margin:0; font-size:1.85rem; font-weight:700;">🚢 Titanic — Exploratory Data Analysis</h1>
  <p style="margin:10px 0 0; opacity:0.92; font-size:1.05rem; line-height:1.45;">
    Question → quality gates → missingness framework → EDA → executive summary.
    Associations only — not causal claims (skill §60.18).
  </p>
</div>

<div style="background:#E8F1F5; border-radius:12px; padding:16px 22px; margin-top:14px;
            font-family: system-ui, sans-serif; border:1px solid #C5D9E3;">
  <p style="margin:0 0 8px; color:#0A3D62; font-weight:600;">Business question</p>
  <p style="margin:0; color:#1B262C; line-height:1.55;">
    Which passenger attributes are <strong>associated</strong> with survival, and what data-quality
    issues must be fixed before modeling? Supports feature prioritization for prediction — not
    causal policy conclusions.
  </p>
</div>


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;"><h2 style="margin:0; font-size:1.25rem; font-weight:600;">1. Setup</h2></div>


In [ ]:
from __future__ import annotations
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid", context="notebook", palette="muted")
plt.rcParams.update({"figure.figsize": (10, 5), "axes.titlecolor": "#0A3D62", "axes.labelcolor": "#1B262C"})

DATA_DIR = Path("../Datasets")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;">
  <h2 style="margin:0; font-size:1.25rem; font-weight:600;">2. Load &amp; Quality Gates (§60.19)</h2>
</div>


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

REQUIRED_TRAIN = ["PassengerId", "Survived", "Pclass", "Name", "Sex", "Age",
                  "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked"]
REQUIRED_TEST = [c for c in REQUIRED_TRAIN if c != "Survived"]

def quality_gates(df: pd.DataFrame, required: list[str], *, has_target: bool) -> None:
    missing_cols = set(required) - set(df.columns)
    assert not missing_cols, f"Missing columns: {missing_cols}"
    assert df["PassengerId"].is_unique and df["PassengerId"].notna().all()
    if has_target:
        assert df["Survived"].isin([0, 1]).all()
    assert df["Pclass"].isin([1, 2, 3]).all()
    assert df["Sex"].isin(["male", "female"]).all()
    print(f"Quality gates passed · shape={df.shape}")

quality_gates(train, REQUIRED_TRAIN, has_target=True)
quality_gates(test, REQUIRED_TEST, has_target=False)
train.head()


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;">
  <h2 style="margin:0; font-size:1.25rem; font-weight:600;">3. Missing Data Framework (§60.17)</h2>
</div>
<div style="background:#FEF5E7; border-left:5px solid #E67E22; border-radius:0 12px 12px 0; padding:14px 18px; font-family:system-ui,sans-serif; margin:12px 0;">
  <strong style="color:#D35400;">Quantify → diagnose mechanism → decide treatment in FE</strong>
</div>


In [ ]:
def missingness_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    miss = df.isnull().sum()
    out = pd.DataFrame({"count": miss, "pct": (miss / len(df) * 100).round(1)})
    out = out[out["count"] > 0].sort_values("count", ascending=False)
    print(f"=== {name} missing ===\n", out if len(out) else "None", "\n")
    return out

missingness_report(train, "Train")
missingness_report(test, "Test")

print("Age missing rate by Pclass:")
print(train.assign(m=train["Age"].isna()).groupby("Pclass")["m"].mean())
print("\nCabin missing rate by Pclass:")
print(train.assign(m=train["Cabin"].isna()).groupby("Pclass")["m"].mean())


<div style="background:#E8F1F5; border-radius:12px; padding:16px 20px; margin:12px 0; font-family:system-ui,sans-serif; border:1px solid #C5D9E3;">
  <p style="margin:0 0 8px; color:#0A3D62; font-weight:600;">Documented assumptions for FE</p>
  <ul style="margin:0; padding-left:1.2rem; color:#1B262C; line-height:1.55;">
    <li><strong>Age (~20%)</strong> — rate varies by class/sex → closer to MAR; conditional impute (Title×Pclass).</li>
    <li><strong>Cabin (~77%)</strong> — tied to class; absence informative → Deck + explicit Missing category.</li>
    <li><strong>Embarked (2)</strong> — mode impute. <strong>Fare (1 in test)</strong> — Pclass median from train.</li>
  </ul>
</div>


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;"><h2 style="margin:0; font-size:1.25rem; font-weight:600;">4. Univariate &amp; Associations</h2></div>


In [ ]:
print(train["Survived"].value_counts(normalize=True))
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=train, x="Survived", hue="Survived", palette=["#C0392B", "#27AE60"], legend=False, ax=ax)
ax.set_title("Survival distribution")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Died (0)", "Survived (1)"])
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=train, x="Sex", y="Survived", hue="Sex", palette=["#3C6382", "#E67E22"], legend=False, errorbar=None, ax=axes[0])
axes[0].set_title("Survival rate by Sex"); axes[0].set_ylim(0, 1)
sns.barplot(data=train, x="Pclass", y="Survived", hue="Pclass", order=[1,2,3], palette="Blues_r", legend=False, errorbar=None, ax=axes[1])
axes[1].set_title("Survival rate by Class"); axes[1].set_ylim(0, 1)
sns.barplot(data=train, x="Pclass", y="Survived", hue="Sex", order=[1,2,3], palette=["#3C6382", "#E67E22"], errorbar=None, ax=axes[2])
axes[2].set_title("Class × Sex"); axes[2].set_ylim(0, 1)
plt.tight_layout(); plt.show()
print(train.groupby(["Sex", "Pclass"])["Survived"].mean().unstack())


In [ ]:
train = train.copy()
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
corr = train[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True, ax=ax)
ax.set_title("Correlation matrix (associations only)")
plt.tight_layout(); plt.show()


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;">
  <h2 style="margin:0; font-size:1.25rem; font-weight:600;">5. Executive Summary (§60.20)</h2>
</div>
<div style="background:#EAF7F0; border-left:5px solid #27AE60; border-radius:0 12px 12px 0; padding:16px 20px; font-family:system-ui,sans-serif; margin:14px 0;">
  <strong style="color:#1E8449;">Findings</strong>
  <ul style="margin:10px 0 0; padding-left:1.25rem; color:#1B262C; line-height:1.65;">
    <li><strong>Association:</strong> Female ~74% survival vs male ~19%; 1st class ~63% vs 3rd ~24%; Sex×Class interaction.</li>
    <li><strong>Data risk:</strong> Age ~20% missing (MAR-leaning); Cabin ~77% missing but informative.</li>
    <li><strong>Limitation:</strong> Observational data — supports prediction, not causal policy claims.</li>
    <li><strong>Recommendation:</strong> Title, Deck, FamilySize; conditional Age impute; baseline models next.</li>
  </ul>
</div>
<div style="background:linear-gradient(135deg,#F8FBFD 0%,#E8F1F5 100%); border:1px solid #B8D0DC; border-radius:12px; padding:16px 20px; font-family:system-ui,sans-serif;">
  <strong style="color:#0A3D62;">Next:</strong> <code>02_feature_engineering.ipynb</code>
</div>
